# Phase 25 — Heterogeneous Information Composition (Phase 17)
## NeuroForge Experimental Research

Phase 15: depth-3 improves isolated R. Phase 16: relational information is genuinely created; embedded computation NOT inferior; suppression/starvation NOT supported. Question: WHERE does information become incompatible when heterogeneous branches must be combined? Target: the composition interface. Distinguish exists / decodable / task-usable / composable — these are not interchangeable claims.

## 1. Phase 16 baseline

In [1]:
import json
p17 = json.load(open('../results/metrics/phase17_composition/summary.json', encoding='utf-8'))
for k in ('baseline_perf_mean','depth3_perf_mean'):
    d = p17[k]
    print(f"{k}: " + '  '.join(f"{f}={d.get(f,0)*100:.1f}%" for f in ('F','R','C','FR','RC','FC','FRC')))
print('Phase 16: relational info created (probe 85%%, drop 15.3pp) but RC/FRC unmoved.')

baseline_perf_mean: F=100.0%  R=57.8%  C=98.9%  FR=76.9%  RC=53.9%  FC=49.4%  FRC=79.4%
depth3_perf_mean: F=100.0%  R=62.5%  C=99.2%  FR=76.7%  RC=53.6%  FC=50.0%  FRC=80.0%
Phase 16: relational info created (probe 85%%, drop 15.3pp) but RC/FRC unmoved.


## 2. Research question

In [2]:
print('Where does information become incompatible when heterogeneous branches combine?')
print('Candidates: interference, destructive addition, scale mismatch, non-separability,')
print('single-state capacity, or task/composition limitation. No new fusion before the gate.')

Where does information become incompatible when heterogeneous branches combine?
Candidates: interference, destructive addition, scale mismatch, non-separability,
single-state capacity, or task/composition limitation. No new fusion before the gate.


## 3. Representation preservation (17B: stages x components)

In [3]:
pres = p17['aggregates']['preservation']['matrix_mean']
for stage, probes in pres.items():
    print(f"{stage:>13}: " + '  '.join(f"{t}={v*100:.1f}%" for t,v in probes.items()))
print()
print('Key Q: does independently-available branch info stay SIMULTANEOUSLY decodable after fusion?')

        input: F_signal=54.2%  R_signal=53.7%  C_signal=64.9%  final_target=55.1%
   feat_delta: F_signal=65.6%  R_signal=55.1%  C_signal=65.0%  final_target=65.0%
    rel_delta: F_signal=68.8%  R_signal=61.4%  C_signal=67.0%  final_target=69.1%
    ctx_delta: F_signal=51.6%  R_signal=54.0%  C_signal=91.2%  final_target=65.1%
  fused_delta: F_signal=64.4%  R_signal=54.9%  C_signal=86.5%  final_target=68.6%
 block_output: F_signal=67.3%  R_signal=59.8%  C_signal=92.0%  final_target=76.2%

Key Q: does independently-available branch info stay SIMULTANEOUSLY decodable after fusion?


## 4. Branch probe matrix (17C: Case 1-4 localization)

In [4]:
for t in ('F_signal','R_signal','C_signal'):
    pre = p17['aggregates']['preservation'][f'branch_{t}']
    post = p17['aggregates']['preservation'][f'fused_{t}']
    print(f"{t}: branch {pre*100:.1f}% -> fused {post*100:.1f}% (drop {(pre-post)*100:+.1f}pp)")
print('Cases: 1=all decodable 2=relational lost 3=present but unusable 4=entangled')

F_signal: branch 65.6% -> fused 67.3% (drop -1.7pp)
R_signal: branch 61.4% -> fused 59.8% (drop +1.5pp)
C_signal: branch 91.2% -> fused 92.0% (drop -0.8pp)
Cases: 1=all decodable 2=relational lost 3=present but unusable 4=entangled


## 5. Scale diagnostics (17F: norms and domination ratios)

In [5]:
import csv
for row in csv.DictReader(open('../results/metrics/phase17_composition/scale_diagnostics.csv')):
    if row['seed'] == str(p17['per_seed_results'][0]['seed']):
        print({k: (round(float(v),4) if (k != 'branch' and v not in ('', None)) else v) for k,v in row.items() if k != 'seed'})
print('Purpose: does one branch numerically dominate the shared residual? (magnitude != importance)')

{'branch': 'feat_delta', 'l2_mean': 3.4846, 'rms': 0.2254, 'mean_abs': 0.1711, 'variance': 0.0507, 'rel_over_feat': '', 'rel_over_ctx': '', 'max_min_branch': ''}
{'branch': 'rel_delta', 'l2_mean': 8.0688, 'rms': 0.4785, 'mean_abs': 0.3989, 'variance': 0.2276, 'rel_over_feat': '', 'rel_over_ctx': '', 'max_min_branch': ''}
{'branch': 'ctx_delta', 'l2_mean': 2.4514, 'rms': 0.1668, 'mean_abs': 0.0364, 'variance': 0.0278, 'rel_over_feat': '', 'rel_over_ctx': '', 'max_min_branch': ''}
{'branch': 'block_output', 'l2_mean': 5.6867, 'rms': 0.3381, 'mean_abs': 0.2689, 'variance': 0.1137, 'rel_over_feat': '', 'rel_over_ctx': '', 'max_min_branch': ''}
{'branch': 'ratios', 'l2_mean': '', 'rms': '', 'mean_abs': '', 'variance': '', 'rel_over_feat': 2.1234, 'rel_over_ctx': 2.8684, 'max_min_branch': 2.8684}
Purpose: does one branch numerically dominate the shared residual? (magnitude != importance)


## 6. Branch interaction (17G: interaction_gain = pair - best single)

In [6]:
import csv
for row in csv.DictReader(open('../results/metrics/phase17_composition/branch_interactions.csv')):
    if row['seed'] == str(p17['per_seed_results'][0]['seed']):
        print(f"{row['pair']:>4} {row['family']}: pair={float(row['pair_acc'])*100:.1f}% gain={float(row['interaction_gain'])*100:+.1f}pp")
print('Negative interaction alone is not proof of interference; combine with probes.')

 F+R R: pair=61.7% gain=+5.8pp
 F+R RC: pair=50.0% gain=-1.7pp
 F+R FRC: pair=70.8% gain=-1.7pp
 R+C R: pair=54.2% gain=+0.8pp
 R+C RC: pair=51.7% gain=+0.0pp
 R+C FRC: pair=80.0% gain=+0.0pp
 F+C R: pair=55.8% gain=+0.0pp
 F+C RC: pair=51.7% gain=+0.0pp
 F+C FRC: pair=80.0% gain=+0.0pp
Negative interaction alone is not proof of interference; combine with probes.


## 7. Composition order (17H: frozen chains; readout confound controlled)

In [7]:
o = p17['aggregates']['order']
print('raw means R:', {k: f'{v*100:.1f}%' for k,v in o['means_R'].items()})
print('SAME-final-expert orientations (readout held fixed):')
for name, d in o['orientations'].items():
    print(f"  {name}: {d['diff_mean']*100:+.1f}pp, positive {d['n_pos']}/{d['n_seeds']} seeds")
print('Raw spread tracks final-expert identity; controlled effects are seed-inconsistent.')

raw means R: {'F→R': '66.7%', 'R→F': '47.5%', 'R→C': '51.4%', 'C→R': '75.6%', 'F→C': '49.2%', 'C→F': '52.8%'}
SAME-final-expert orientations (readout held fixed):
  C-first_graph-final: +8.9pp, positive 2/3 seeds
  C-first_mlp-final: +5.3pp, positive 2/3 seeds
  R-first_v2-final: +2.2pp, positive 2/3 seeds
Raw spread tracks final-expert identity; controlled effects are seed-inconsistent.


## 8. Oracle representation (17E: full branch info vs fused state, one protocol)

In [8]:
suff = p17['aggregates']['sufficiency']
print(f"oracle R/RC/FRC: {suff['oracle_R']*100:.1f}/{suff['oracle_RC']*100:.1f}/{suff['oracle_FRC']*100:.1f}%")
print(f"gains over fused head: RC {suff['oracle_RC_gain']*100:+.1f}pp, FRC {suff['oracle_FRC_gain']*100:+.1f}pp")
print(f"causal drops: oracle {suff['oracle_drop_R']*100:+.1f}pp vs fused {suff['fused_drop_R']*100:+.1f}pp")

oracle R/RC/FRC: 59.7/53.3/76.1%
gains over fused head: RC +0.3pp, FRC +6.1pp
causal drops: oracle +4.7pp vs fused +5.6pp


## 9. Joint decodability (17J: R-only vs R+C vs F+R+C on RC/FRC)

In [9]:
import csv
for row in csv.DictReader(open('../results/metrics/phase17_composition/joint_decodability.csv')):
    if row['seed'] == str(p17['per_seed_results'][0]['seed']):
        print(f"{row['info']:>6}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")
print('R good + R+C bad = interference. R+C improves = coexistence. None improve = deeper limit.')

     R: R=80.0% RC=46.7% FRC=55.0%
   R+C: R=69.2% RC=50.8% FRC=79.2%
 F+R+C: R=65.8% RC=50.8% FRC=80.0%
R good + R+C bad = interference. R+C improves = coexistence. None improve = deeper limit.


## 10. Frozen diagnostic composition (17K: diagnostic head vs production head)

In [10]:
import csv
for row in csv.DictReader(open('../results/metrics/phase17_composition/frozen_oracle.csv')):
    if row['seed'] == str(p17['per_seed_results'][0]['seed']):
        print(f"{row['head']:>11}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")

oracle_diag: R=65.8% RC=50.8% FRC=80.0%
 fused_diag: R=73.3% RC=48.3% FRC=77.5%
 production: R=60.0% RC=51.7% FRC=80.0%


## 11. RC/FRC analysis (17M primary criteria + agreement splits)

In [11]:
import csv
for row in csv.DictReader(open('../results/metrics/phase17_composition/component_agreement.csv')):
    if row['seed'] == str(p17['per_seed_results'][0]['seed']):
        print(f"{row['family']} {row['split']}: acc={float(row['acc'])*100:.1f}% (n={row['n']})")
print('RC-disagree ~0%% means: knows components, cannot combine them.')

RC agree: acc=100.0% (n=61)
RC disagree: acc=1.7% (n=59)
FRC agree: acc=100.0% (n=33)
FRC disagree: acc=72.4% (n=87)
RC-disagree ~0%% means: knows components, cannot combine them.


## 12. Causal controls (destruction alongside absolute accuracy)

In [12]:
import csv
for row in csv.DictReader(open('../results/metrics/phase17_composition/causal_controls.csv')):
    if row['seed'] == str(p17['per_seed_results'][0]['seed']):
        print(f"{row['condition']:>9} {row['variant']:>20}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")

 baseline             original: R=50.0% RC=50.8% FRC=80.0%
 baseline  relational_permuted: R=48.3% RC=50.8% FRC=80.0%
   depth3             original: R=60.0% RC=51.7% FRC=80.0%
   depth3  relational_permuted: R=56.7% RC=50.8% FRC=80.0%


## 13. Efficiency (params/FLOPs/latency/activation; FLOPs != latency)

In [13]:
import csv
for name in ('compute.csv','latency.csv'):
    print(f'--- {name} ---')
    for row in csv.DictReader(open('../results/metrics/phase17_composition/' + name)):
        if row['seed'] == str(p17['per_seed_results'][0]['seed']):
            print(' ', {k: v for k,v in row.items() if k != 'seed'})

--- compute.csv ---
  {'condition': 'graph', 'params': '3866', 'rel_params': '1776'}
  {'condition': 'baseline', 'params': '6464', 'rel_params': '1776'}
  {'condition': 'depth3', 'params': '10016', 'rel_params': '5328'}
  {'condition': 'rel_first', 'params': '10016', 'rel_params': '5328'}
--- latency.csv ---
  {'condition': 'graph', 'latency_us': '38.3194166655206', 'throughput_per_s': '26096.430661476883'}
  {'condition': 'baseline', 'latency_us': '62.12833333241482', 'throughput_per_s': '16095.7158572007'}
  {'condition': 'depth3', 'latency_us': '85.07350001309533', 'throughput_per_s': '11754.541659224911'}
  {'condition': 'rel_first', 'latency_us': '96.25991666932048', 'throughput_per_s': '10388.540054894058'}


## 14. H1-H8 (re-derived programmatically)

In [14]:
from neuroforge.evaluation.phase17_composition_diagnostics import build_phase17_hypotheses
hyps = build_phase17_hypotheses(p17['aggregates'])
for h in sorted(hyps):
    print(f"{h}: {hyps[h]['status']}")
    print(f"    {hyps[h]['evidence']}")
assert all(hyps[h]['status'] == p17['hypotheses'][h]['status'] for h in hyps)
print('stored verdicts match fresh derivation: OK')

H1: NOT SUPPORTED
    Branch information remains decodable after fusion (worst drop -1.7pp).
H2: PARTIALLY SUPPORTED
    Max/min branch RMS ratio 2.40x (moderate imbalance).
H3: INCONCLUSIVE
    Weak negative interaction -2.5pp.
H4: PARTIALLY SUPPORTED
    Same-final-expert orientation C-first_graph-final +8.9pp on R but seed-inconsistent (2/3); raw spread 28.1pp is dominated by final-expert identity.
H5: PARTIALLY SUPPORTED
    Oracle RC 53.3% (+0.3pp), FRC 76.1% (+6.1pp) (modest).
H6: SUPPORTED
    Diagnostic readout exploits jointly available info (RC +0.3pp, FRC +6.1pp) that fusion/readout does not.
H7: NOT TESTED
    No composition mechanism earned an intervention.
H8: NOT TESTED
    No validated candidate.
stored verdicts match fresh derivation: OK


## 15. Final CASE (programmatic)

In [15]:
from neuroforge.evaluation.phase17_composition_diagnostics import select_phase17_case
case, label = select_phase17_case(hyps, p17['aggregates'])
print(f'Programmatic verdict: {case} — {label}')
assert case == p17['verdict_case']
print(f"Minimal intervention: {p17['minimal_intervention']['intervention']} ({p17['minimal_intervention']['outcome']})")

Programmatic verdict: CASE E — Branch information exists but is not jointly task-usable
Minimal intervention: none (NO INTERVENTION (diagnosis only))


## 16. Evidence-backed next step (programmatic recommendation)

In [16]:
from neuroforge.evaluation.phase17_composition_diagnostics import recommendation_for_case
print('Recommendation:', recommendation_for_case(p17['verdict_case']))
print()
print('Loaded (not typed):')
print(f"  preservation worst drop: {min(p17['aggregates']['preservation'][t] for t in ('F_signal_drop_pre_to_fused','R_signal_drop_pre_to_fused','C_signal_drop_pre_to_fused'))*100:+.1f}pp")
print(f"  oracle RC/FRC gains: {p17['aggregates']['sufficiency']['oracle_RC_gain']*100:+.1f}/{p17['aggregates']['sufficiency']['oracle_FRC_gain']*100:+.1f}pp")
print(f"  domination ratio: {p17['aggregates']['domination']['max_min_branch_mean']:.2f}x")

Recommendation: Treat joint usability as the open problem; isolated gains do not compose.

Loaded (not typed):
  preservation worst drop: -1.7pp
  oracle RC/FRC gains: +0.3/+6.1pp
  domination ratio: 2.40x
